# Ten stationary benchmarks

The negative control: objectives with no intended variation of local
regularity. Runtime is about fifteen minutes.

In [ ]:
import sys, time, pickle
sys.path.insert(0, '..')          # drop this line if the notebook is at the root

import numpy as np
from joblib import Parallel, delayed

from ibo import (sigma_estimator, get_automatic_kernel_args,
                 evaluate_single_bo, kim1,
                 get_benchmarks, shift_benchmarks,
                 sample_gp_function_, generate_centers,
                 plot_grouped_comparison, plot_mean_ratio_across_functions,
                 report_final_gap, paired_comparison)

The minimisers are displaced before the length-scales are fitted, so that
the kernel of a benchmark is estimated on the function actually optimised,
and so that no optimum sits on a cut of the fixed grid of $\mathrm{IBO}_4$.

In [ ]:
functions_bench, argmin_locations = get_benchmarks()
functions_bench, argmin_locations = shift_benchmarks(functions_bench, argmin_locations)

bounds_bench = np.array([[-1, 1], [-1, 1]])
d = bounds_bench.shape[0]

tree_depth = 3
n_init     = 40 * d
n_iter     = 200
n_seeds    = 30
min_leaf   = 16
tau        = 1.5          # splitting threshold, shared across all suites
noise_var  = 1e-8

In [ ]:
kernel_args_per_function = get_automatic_kernel_args(
    functions_bench, d=d, n_points=n_init + n_iter, bounds=(-1.0, 1.0))
kernel_args_per_function

In [ ]:
configs = [
    # --- EI Group ---
    ("Classic BO - EI", 'fixed', 1,  False, 'ei', {'xi': 0.0}),
    ("IBO_1 - EI",      'fixed', 1,  True,  'ei', {'xi': 0.0}),
    ("IBO_4 - EI",      'fixed', 2,  True,  'ei', {'xi': 0.0}),
    ("IBO_P - EI",      'tree',  2,  True,  'ei', {'xi': 0.0}),

    # --- PI Group ---
    ("Classic BO - PI", 'fixed', 1,  False, 'pi', {'xi': 0.0}),
    ("IBO_1 - PI",      'fixed', 1,  True,  'pi', {'xi': 0.0}),
    ("IBO_4 - PI",      'fixed', 2,  True,  'pi', {'xi': 0.0}),
    ("IBO_P - PI",      'tree',  2,  True,  'pi', {'xi': 0.0}),

    # --- LCB Group ---
    ("Classic BO - LCB", 'fixed', 1, False, 'lcb', {'beta': 1.96}),
    ("IBO_1 - LCB",      'fixed', 1, True,  'lcb', {'beta': 1.96}),
    ("IBO_4 - LCB",      'fixed', 2, True,  'lcb', {'beta': 1.96}),
    ("IBO_P - LCB",      'tree',  2, True,  'lcb', {'beta': 1.96}),

    # --- TS Group ---
    ("Classic BO - TS", 'fixed', 1, False, 'ts', {}),
    ("IBO_1 - TS",      'fixed', 1, True,  'ts', {}),
    ("IBO_4 - TS",      'fixed', 2, True,  'ts', {}),
    ("IBO_P - TS",      'tree',  2, True,  'ts', {}),
]

In [ ]:
all_histories_bench = {}
leaf_report = {}
start = time.perf_counter()

for f_id, (f_name, objective_function) in enumerate(functions_bench.items()):
    print(f"\nRunning {f_name} ({f_id+1}/{len(functions_bench)}) in parallel...")

    results_bench = Parallel(n_jobs=-1)(
        delayed(evaluate_single_bo)(
            seed, objective_function, bounds_bench, configs, n_init, d, n_iter,
            kernel_args_per_function[f_name], noise_var, tree_depth,
            sigma_estimator, tau=tau, n_candidates=128, min_leaf=min_leaf,
            return_info=True,
        ) for seed in range(n_seeds)
    )

    histories_ = {c[0]: np.zeros((n_seeds, n_iter + 1)) for c in configs}
    for seed, seed_histories, infos in results_bench:
        for label, history in seed_histories.items():
            histories_[label][seed, :] = history
    all_histories_bench[f_name] = histories_

    # On a stationary objective the tree should not split: record how often it
    # does, as a check on the splitting criterion and on tau.
    leaves = [info["n_leaves_hist"].mean()
              for _, _, infos in results_bench
              for label, info in infos.items() if label.startswith("IBO_P")]
    leaf_report[f_name] = (float(np.mean(leaves)),
                           float(np.mean([l > 1 for l in leaves])))
    print(f"   IBO_P mean leaves {leaf_report[f_name][0]:.2f}, "
          f"split in {100*leaf_report[f_name][1]:.0f}% of runs")

print(f"\nExecution time: {time.perf_counter() - start:.1f} s")
print(f"IBO_P across the suite (tau = {tau}): "
      f"mean leaves {np.mean([v[0] for v in leaf_report.values()]):.2f}, "
      f"split in {100*np.mean([v[1] for v in leaf_report.values()]):.0f}% of runs")

In [ ]:
plot_mean_ratio_across_functions(all_histories_bench,
                                save_path='Benchmarks_mean_ratio.pdf')

In [ ]:
report_final_gap(all_histories_bench, f_min=1.0)

The paired differences are taken on the normalised regret, so that
functions whose scales differ by orders of magnitude contribute equally.

In [ ]:
nr = {n: {lab: (np.asarray(h, float) - 1.0) / (np.asarray(h, float)[:, [0]] - 1.0)
          for lab, h in hs.items()}
      for n, hs in all_histories_bench.items()}
_ = paired_comparison(nr, f_min=0.0)

In [ ]:
tag = f"tau{tau}_leaf{min_leaf}_init{n_init}_iter{n_iter}"
with open(f'../results/bench_{tag}.pkl', 'wb') as fh:
    pickle.dump({'histories': all_histories_bench,
                 'leaves': leaf_report,
                 'kernel_args': kernel_args_per_function,
                 'params': {'tau': tau, 'min_leaf': min_leaf, 'n_init': n_init,
                            'n_iter': n_iter, 'n_seeds': n_seeds,
                            'tree_depth': tree_depth}}, fh)
print(f'saved bench_{tag}.pkl')